<a href="https://colab.research.google.com/github/zero0r0/GetSongSetListTool/blob/main/SONG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 2. ついでに軽量なJavaScriptエンジン「Deno」もインストール（yt-dlpが推奨しているもの）
!curl -fsSL https://deno.land/x/install/install.sh | sh
import os
os.environ['PATH'] += ":/root/.deno/bin"

# 3. ライブラリを最新に
!pip install -U google-genai yt-dlp

In [ ]:
# @title 歌枠セトリ自動生成ツール
# @markdown **【準備】**<br>
# @markdown 1. 左の鍵マーク🔑(シークレット)に `GEMINI_API_KEY` を登録してください。<br>
# @markdown 2. 下記にYouTubeのURLを入力し、左の実行ボタン（▶）を押してください。

youtube_url = "" # @param {type:"string"}
target_model = "gemini-2.5-flash" # @param ["gemini-3-flash-preview", "gemini-2.5-flash", "gemini-2.0-flash", "gemini-1.5-flash"]

import os
import time
import math
import subprocess
import re
import difflib
import yt_dlp
from google import genai
from google.colab import userdata, files

def check_model_availability(client, model_name):
    """選択されたモデルが現在利用可能かチェックする"""
    print(f"🔍 モデル '{model_name}' の有効性を確認中...")
    try:
        available_models = [m.name for m in client.models.list()]

        # APIからは 'models/gemini-2.5-flash' などの形式で返ってくるため、部分一致で判定
        is_valid = any(model_name in m for m in available_models)

        if is_valid:
            print("✅ モデルの確認が完了しました。")
            return True
        else:
            print(f"❌ エラー: モデル '{model_name}' は現在利用できません。")
            print("💡 利用可能なFlashモデルの例:")
            for m in available_models:
                if "flash" in m.lower():
                    print(f" - {m.replace('models/', '')}")
            return False
    except Exception as e:
        print(f"⚠️ モデルの確認中にAPIエラーが発生しました: {e}")
        return False

def download_full_audio(url, output_name="full_audio"):
    """動画全体を一度だけダウンロードする"""
    print(f"\n📥 動画全体の音声をダウンロード中... (これには少し時間がかかります)")
    output_filename = f"{output_name}.m4a"

    if os.path.exists(output_filename):
        os.remove(output_filename)

    cookie_file = "cookies.txt"
    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': output_name,
        'postprocessors': [{'key': 'FFmpegExtractAudio', 'preferredcodec': 'm4a'}],
        'quiet': True,
        'no_warnings': True,
        'user_agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'referer': 'https://www.google.com/',
    }

    if os.path.exists(cookie_file):
        print(f"🍪 {cookie_file} を使用して認証を試みます")
        ydl_opts['cookiefile'] = cookie_file

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
        print("✅ ダウンロード完了！")
        return output_filename
    except Exception as e:
        print(f"❌ ダウンロードエラー: {e}")
        print("💡 ヒント: cookies.txtをアップロードすると解決する場合があります。")
        return None

def split_audio_local(input_file, start_time, end_time, output_file):
    """音声を指定した時間でローカル分割する"""
    command = [
        'ffmpeg', '-y',
        '-ss', str(start_time),
        '-to', str(end_time),
        '-i', input_file,
        '-c', 'copy',
        '-loglevel', 'error',
        output_file
    ]
    subprocess.run(command, check=True)
    return output_file

def analyze_part(client, file_path, base_offset_seconds, model_name):
    """分割した音声をGeminiで解析する"""
    print(f"🤖 モデル: {model_name}")
    print(f"      📤 AIに送信中...", end="")
    uploaded_file = client.files.upload(file=file_path)

    while uploaded_file.state.name == "PROCESSING":
        time.sleep(2)
        uploaded_file = client.files.get(name=uploaded_file.name)
    print(" [完了] → 🧠 解析中...")

    offset_str = str(time.strftime('%H:%M:%S', time.gmtime(base_offset_seconds)))
    prompt = f"""
    この音声ファイルを解析して、歌枠のセットリストを作成してください。
    【ルール】
    1. 各曲の開始時間を【00:00】曲名 / アーティスト名の形式で示してください。
    2. この音声は動画の開始から {offset_str} 経過した時点からのものです。
       出力するタイムスタンプは、この時間を加算して動画全体での正しい時間を出力してください。
    3. 曲名 / アーティスト名 で表記してください
    4. 歌っていないお喋り部分は除外してください。
    5. 曲が見つからない場合は特になにも出力しなくてよいです。
    """

    try:
        response = client.models.generate_content(model=model_name, contents=[prompt, uploaded_file])
        return response.text
    finally:
        try: client.files.delete(name=uploaded_file.name)
        except: pass

def time_to_seconds(time_str):
    """文字列の時間を秒に変換する便利関数"""
    parts = list(map(int, time_str.split(':')))
    if len(parts) == 2:
        return parts[0] * 60 + parts[1]
    elif len(parts) == 3:
        return parts[0] * 3600 + parts[1] * 60 + parts[2]
    return 0

def format_and_deduplicate(results):
    """各セグメントのテキストを結合し、境界の重複を排除して綺麗なリストにする"""
    print("\n🧹 セグメント間の重複排除とフォーマット整形を実行中...")
    all_lines = "\n".join(results).split('\n')

    cleaned_lines = []
    last_seconds = -9999
    last_title = ""

    for line in all_lines:
        line = line.strip()
        if not line: continue

        match = re.search(r'【(\d{1,2}:\d{2}(?::\d{2})?)】(.*)', line)
        if match:
            time_str = match.group(1)
            title_str = match.group(2).strip()
            current_seconds = time_to_seconds(time_str)

            similarity = difflib.SequenceMatcher(None, last_title, title_str).ratio()

            # 5分以内 かつ 類似度50%以上の場合は重複とみなす
            if abs(current_seconds - last_seconds) < 300 and similarity > 0.5:
                print(f"🔄 重複を排除しました: {line} (類似度: {similarity:.2f})")
                continue

            cleaned_line = f"【{time_str}】 {title_str}"
            cleaned_lines.append(cleaned_line)

            last_seconds = current_seconds
            last_title = title_str

    return "\n".join(cleaned_lines)

def validate_and_correct_setlist(client, setlist_text, model_name):
    """Geminiを使って、最終的なセットリストの曲名・アーティスト名の表記を修正する"""
    print(f"\n✨ 最終工程: Gemini ({model_name}) で曲名・アーティスト名の表記を校正中...")

    prompt = f"""
    以下のセットリストの「曲名」と「アーティスト名」の組み合わせをチェックし、間違いや表記ゆれ（誤字脱字など）があれば修正してください。

    【厳守するルール】
    1. タイムスタンプ（例: 【01:23:45】）は絶対にそのまま保持してください。時間はいじらないでください。
    2. フォーマットは必ず「【タイムスタンプ】 曲名 / アーティスト名」を維持してください。
    3. 明らかな間違い（例: 「紅蓮華 / リサ」→「紅蓮華 / LiSA」や、「アイドル / YOASOBI」の誤字など）のみ修正してください。
    4. アニメのキャラクター名義やVtuberのオリジナル曲など、判断が難しいマイナーな曲は元のテキストをそのまま残してください。
    5. 「修正しました」「以下がセットリストです」などの余計なテキストは一切出力せず、セットリスト本体のテキストのみを出力してください。

    【対象のセットリスト】
    {setlist_text}
    """

    try:
        response = client.models.generate_content(model=model_name, contents=prompt)
        print("✅ 校正完了！")
        return response.text.strip()
    except Exception as e:
        print(f"⚠️ 最終チェック中にエラーが発生しました。元のリストをそのまま使用します。: {e}")
        return setlist_text

def main():
    if not youtube_url:
        print("❌ URLが入力されていません。")
        return

    try:
        client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
    except userdata.SecretNotFoundError:
        print("❌ シークレットに 'GEMINI_API_KEY' が設定されていません。左の🔑アイコンから登録してください。")
        return

    # モデルの有効性チェック (フェイルファスト)
    if not check_model_availability(client, target_model):
        return

    info_opts = {'quiet': True}
    if os.path.exists("cookies.txt"): info_opts['cookiefile'] = "cookies.txt"

    try:
        with yt_dlp.YoutubeDL(info_opts) as ydl:
            info = ydl.extract_info(youtube_url, download=False)
            duration = info['duration']
            title = info['title']
    except Exception as e:
        print(f"❌ 情報取得エラー: {e}")
        return

    print(f"\n🎬 {title} (総時間: {math.ceil(duration/60)}分)")

    full_audio_path = download_full_audio(youtube_url)
    if not full_audio_path: return

    results = []
    interval = 1800 # 30分ごとに分割
    num_segments = math.ceil(duration / interval)

    print("\n🚀 ローカル分割解析を開始します...")

    for i in range(num_segments):
        start = i * interval
        end = min((i + 1) * interval, duration)
        print(f"\n📂 [{i+1}/{num_segments}] セグメント ({math.ceil(start/60)}分〜{math.ceil(end/60)}分)")
        segment_file = f"part_{i}.m4a"
        split_audio_local(full_audio_path, start, end, segment_file)

        try:
            res_text = analyze_part(client, segment_file, start, target_model)
            print(f"📝 結果:\n{res_text}\n" + "-"*30)
            results.append(res_text)
        except Exception as e:
            print(f"⚠️ エラー: {e}")
        finally:
            if os.path.exists(segment_file): os.remove(segment_file)

    if os.path.exists(full_audio_path): os.remove(full_audio_path)

    # 取得した全セグメントの結果を重複排除・整形
    cleaned_output = format_and_deduplicate(results)

    # 最後にGeminiによる表記の校正を実行
    final_output = validate_and_correct_setlist(client, cleaned_output, target_model)

    # テキストファイルとして保存・ダウンロード
    with open("setlist.txt", "w", encoding="utf-8") as f:
        f.write(f"TITLE: {title}\nURL: {youtube_url}\n\n{final_output}")

    print("\n🎉 すべての処理が完了しました！ setlist.txt をダウンロードします。")
    files.download("setlist.txt")

if __name__ == "__main__":
    main()